In [ ]:
!pip install -q datasets transformers accelerate evaluate scikit-learn

In [ ]:
from datasets import load_dataset, Dataset
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer

raw_datasets = load_dataset("ailsntua/QEvasion")

model_name = "answerdotai/ModernBERT-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

print(f"ModernBERT max length: {tokenizer.model_max_length}")

train_df = raw_datasets['train'].to_pandas()

train_df_split, val_df_split = train_test_split(
    train_df,
    test_size=700,
    random_state=42,
    stratify=train_df['clarity_label']
)

raw_datasets['train'] = Dataset.from_pandas(train_df_split, preserve_index=False)
raw_datasets['validation'] = Dataset.from_pandas(val_df_split, preserve_index=False)

label2id = {
    "Clear Reply": 0,
    "Ambivalent": 1,
    "Clear Non-Reply": 2
}

id2label = {v: k for k, v in label2id.items()}

print(raw_datasets)

In [ ]:
from transformers import DataCollatorWithPadding

def tokenize_function(examples):
    inputs = [
        f"Question: {q}\nAnswer: {a}"
        for q, a in zip(examples["question"], examples["interview_answer"])
    ]
    tokenized = tokenizer(
        inputs,
        truncation=True,
        max_length=3000,
        padding=False
    )
    tokenized["labels"] = [label2id[l] for l in examples["clarity_label"]]
    return tokenized

tokenized_datasets = {}
tokenized_datasets['train'] = raw_datasets['train'].map(
    tokenize_function,
    batched=True,
    remove_columns=raw_datasets['train'].column_names
)
tokenized_datasets['validation'] = raw_datasets['validation'].map(
    tokenize_function,
    batched=True,
    remove_columns=raw_datasets['validation'].column_names
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average='macro')

    return {
        'accuracy': accuracy,
        'f1': f1
    }

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results_modernbert",
    per_device_train_batch_size=6,
    gradient_accumulation_steps=8,
    per_device_eval_batch_size=4,
    learning_rate=3e-5,
    weight_decay=0.1,
    warmup_ratio=0.06,
    num_train_epochs=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_dir="./logs",
    logging_steps=50,
    seed=42,
    fp16=True,
    optim="adamw_torch_fused",
    report_to="none",
)


In [ ]:
from transformers import Trainer, EarlyStoppingCallback

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

print("Fine tuning complete.")

In [ ]:
print("Subtask 1: Clarity Classification - Direct Prediction\n")

from datasets import load_dataset
from sklearn.metrics import precision_recall_fscore_support, f1_score, classification_report, confusion_matrix, accuracy_score

test_dataset_raw = load_dataset("ailsntua/QEvasion")['test']

def tokenize_test(examples):
    texts = [
        f"Question: {q} Answer: {a}"
        for q, a in zip(examples['question'], examples['interview_answer'])
    ]
    return tokenizer(texts, truncation=True, padding=False, max_length=8192)

test_tokenized = test_dataset_raw.map(
    tokenize_test,
    batched=True,
    remove_columns=[col for col in test_dataset_raw.column_names
                    if col not in ['clarity_label']]
)

predictions_logits = trainer.predict(test_tokenized)
predictions = np.argmax(predictions_logits.predictions, axis=-1)

predicted_labels = [id2label[pred] for pred in predictions]

y_true_clarity = [test_dataset_raw[i]['clarity_label'] for i in range(len(test_dataset_raw))]
y_pred_clarity = predicted_labels

accuracy_clarity = accuracy_score(y_true_clarity, y_pred_clarity)

clarity_labels = ["Clear Reply", "Ambivalent", "Clear Non-Reply"]
macro_f1_clarity = f1_score(y_true_clarity, y_pred_clarity, average='macro', labels=clarity_labels, zero_division=0)

print(f"Accuracy: {accuracy_clarity:.4f}")
print(f"Macro F1: {macro_f1_clarity:.4f}")

print("\nPer-class metrics:")
print(classification_report(y_true_clarity, y_pred_clarity, labels=clarity_labels, zero_division=0))

print("\nConfusion Matrix:")
cm = confusion_matrix(y_true_clarity, y_pred_clarity, labels=clarity_labels)
print(f"Labels order: {clarity_labels}")
print(cm)